In [11]:


import os
import re
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset


from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# OpenRouter
from openai import OpenAI

# UI
import gradio as gr
from dotenv import load_dotenv
load_dotenv(override=True)




True

### LOad Documents 

In [13]:
DATA_PATH = "data"

documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print("Total pages loaded:", len(documents))

Total pages loaded: 429


### Add Metadata (Very Important)

This allows the AI to cite page numbers and document names.

In [14]:
for doc in documents:
    doc.metadata["source"] = doc.metadata.get("source", "unknown")

### Split Documents into Chunks

This improves retrieval accuracy.

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 1352


### Create Embeddings

We will use embeddings from Hugging Face.

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Create Vector Database

Using Chroma.

In [17]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    chunks,
    embedding_model,
    persist_directory="legal_vector_db"
)

vector_db.persist()

### Retrieval Function

In [18]:
def retrieve_context(question):

    docs = vector_db.similarity_search(question, k=4)

    context = ""
    sources = []

    for doc in docs:
        context += doc.page_content + "\n\n"

        source = f"{doc.metadata.get('source')} | page {doc.metadata.get('page')}"
        sources.append(source)

    return context, sources

### Connect to OpenRouter

We call an LLM through OpenRouter.

In [19]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [20]:
def ask_llm(question):

    context, sources = retrieve_context(question)

    prompt = f"""
You are an expert Nigerian electoral law assistant.

Use ONLY the context provided to answer.

If the answer is not in the context, say:
"I could not find this in the electoral documents."

Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content

    sources_text = "\n".join(set(sources))

    return f"{answer}\n\nSources:\n{sources_text}"

In [23]:
chat_history = []

In [24]:
def chat(question):

    response = ask_llm(question)

    chat_history.append((question, response))

    return response

In [25]:
import gradio as gr

interface = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Ask a question about electoral law..."
    ),
    outputs="text",
    title="Nigerian Electoral Law AI Assistant",
    description="Ask questions about the Electoral Act and Constitution"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
